# Intelligent Signal Processing — Final Term Coursework
## Exercise 2: Rice Coding (K=2 and K=4) — Lossless Audio Compression

### Aim
This notebook implements a **lossless** audio compression system using **Rice coding** (a special case of Golomb coding).  
For each input WAV file (Sound1.wav and Sound2.wav), I:

1. Read the WAV header and PCM samples.
2. Apply a simple **predictive model** (previous-sample predictor) and encode **residuals**.
3. Map signed residuals to unsigned integers (zigzag mapping).
4. Rice-encode the mapped integers into a compact bitstream using a chosen parameter **K**.
5. Save the result as a custom `.ex2` file.
6. Decode the `.ex2` file back to a WAV and verify **byte-identical** reconstruction.
7. Compare compression performance for **K = 2** vs **K = 4**.

The implementation is designed to match the coursework requirement for producing:
`Sound1.wav → Sound1_Enc.ex2 → Sound1_Enc_Dec.wav` (and similarly for Sound2).

In [ ]:
import wave
import os
import struct
from dataclasses import dataclass

## 1) Reading WAV input (metadata + raw PCM frames)

I read the WAV file using Python’s `wave` module and store key metadata:
- `nchannels` (mono/stereo),
- `sampwidth` (bytes per sample),
- `framerate` (sample rate),
- `nframes` (number of frames).

The function `read_wav()` returns:
- the header information (`WavInfo`), and
- the raw PCM byte stream (`frames`) exactly as stored in the file.

This metadata is also written into the `.ex2` header so the decoder can reconstruct an identical WAV format.

In [ ]:
@dataclass
class WavInfo:
    nchannels: int
    sampwidth: int
    framerate: int
    nframes: int
    comptype: str
    compname: str

def read_wav(path):
    with wave.open(path, 'rb') as w:
        info = WavInfo(
            nchannels=w.getnchannels(),
            sampwidth=w.getsampwidth(),
            framerate=w.getframerate(),
            nframes=w.getnframes(),
            comptype=w.getcomptype(),
            compname=w.getcompname()
        )
        frames = w.readframes(info.nframes)  # raw PCM bytes
    return info, frames

## 2) PCM conversion: bytes ↔ integer samples

The encoder works on integer samples, so I convert the raw PCM bytes into signed integers.

Supported sample widths:
- **8-bit PCM**: usually stored as unsigned (0–255), so I convert to signed by subtracting 128.
- **16-bit PCM**: unpacked using little-endian signed short (`<h`).
- **32-bit PCM**: unpacked using little-endian signed int (`<i`).

The inverse conversion (`samples_to_bytes`) is used during decoding to write the reconstructed integer samples back into a valid WAV file.

In [ ]:
def bytes_to_samples(frames, sampwidth):
    if sampwidth == 1:
        # 8-bit PCM is usually unsigned in WAV, convert to signed centered at 0
        samples = list(frames)
        samples = [s - 128 for s in samples]
        return samples

    if sampwidth == 2:
        fmt = "<" + "h" * (len(frames)//2)  # signed 16-bit
        return list(struct.unpack(fmt, frames))

    if sampwidth == 4:
        fmt = "<" + "i" * (len(frames)//4)  # signed 32-bit
        return list(struct.unpack(fmt, frames))

    raise ValueError("Unsupported sample width")

In [ ]:
def samples_to_bytes(samples, sampwidth):
    if sampwidth == 1:
        # signed back to unsigned
        out = bytes([(s + 128) & 0xFF for s in samples])
        return out

    if sampwidth == 2:
        fmt = "<" + "h" * len(samples)
        return struct.pack(fmt, *samples)

    if sampwidth == 4:
        fmt = "<" + "i" * len(samples)
        return struct.pack(fmt, *samples)

    raise ValueError("Unsupported sample width")

## 3) Predictive coding: residuals and reconstruction

To improve compression, I encode **residuals** instead of raw samples.

Prediction model:
- `predicted_sample[i] = sample[i-1]` (previous-sample predictor)

Residual definition:
- `residual[0] = sample[0]` (first sample stored directly)
- `residual[i] = sample[i] - sample[i-1]` for `i ≥ 1`

Decoding is the exact inverse (cumulative sum):
- `sample[0] = residual[0]`
- `sample[i] = sample[i-1] + residual[i]`

This makes the system **lossless** as long as residuals are decoded correctly. This simple first-order predictor is computationally efficient and works well for smoothly varying signals, which makes it suitable for real-time or embedded compression systems.

In [ ]:
def residuals(samples):
    if not samples:
        return []
    res = [samples[0]]
    for i in range(1, len(samples)):
        res.append(samples[i] - samples[i-1])
    return res

def reconstruct_from_residuals(res):
    if not res:
        return []
    samples = [res[0]]
    for i in range(1, len(res)):
        samples.append(samples[-1] + res[i])
    return samples

## 4) Signed-to-unsigned mapping (zigzag)

Rice coding operates on **non-negative integers**, but residuals can be negative.

I therefore apply a zigzag-style mapping so that small-magnitude residuals (both positive and negative) become small unsigned values:

- `0 → 0`
- `-1 → 1`
- `+1 → 2`
- `-2 → 3`
- `+2 → 4`
- ...

In the code, I use the helper functions:
- `zz_enc(x)` for encoding,
- `zz_dec(u)` for decoding.

This mapping is essential for good compression because audio residuals are typically concentrated near 0.

In [ ]:
def zz_enc(x):
    return 2*x if x >= 0 else -2*x - 1

def zz_dec(u):
    return u//2 if (u % 2 == 0) else -(u//2) - 1

## 5) Bitstream handling (BitWriter / BitReader)

Rice coding produces a variable-length code (unary + fixed remainder), so I store the output in a compact **bitstream**.

### `BitWriter`
- Accumulates bits into bytes (`bytearray`)
- Keeps a current byte (`curr`) and bit count (`nbits`)
- `write_ones(n)` efficiently writes `n` consecutive 1-bits (used by unary coding)
- `to_bytes()` flushes any partially-filled byte (padding with zeros) and returns:
  - the byte payload, and
  - the exact number of valid bits (`total_bits`)

### `BitReader`
- Reads individual bits in MSB-first order from the payload
- Reconstructs integers by reading unary + remainder bits

Storing `bit_count` in the `.ex2` header ensures the decoder knows the intended length of the bitstream.

In [ ]:
class BitWriter:
    def __init__(self):
        self.buf = bytearray()
        self.curr = 0
        self.nbits = 0
        self.total_bits = 0

    def write_bit(self, b: int):
        self.curr = (self.curr << 1) | (1 if b else 0)
        self.nbits += 1
        self.total_bits += 1

        if self.nbits == 8:
            self.buf.append(self.curr)
            self.curr = 0
            self.nbits = 0

    def write_bits(self, value: int, n: int):
        for i in range(n - 1, -1, -1):
            self.write_bit((value >> i) & 1)


    def write_ones(self, n: int):
        # Write n bits of value 1 efficiently
        while n > 0:
            space = 8 - self.nbits

            if n >= space:
                 # Fill the current byte with ones
                 self.curr = (self.curr << space) | ((1 << space) - 1)
                 self.nbits += space
                 self.total_bits += space
                 n -= space

                 # Flush full byte
                 if self.nbits == 8:
                     self.buf.append(self.curr)
                     self.curr = 0
                     self.nbits = 0
            else:
                # Partially fill with ones
                self.curr = (self.curr << n) | ((1 << n) - 1)
                self.nbits += n
                self.total_bits += n
                n = 0

    def finish(self):
        if self.nbits != 0:
            self.curr = self.curr << (8 - self.nbits)
            self.buf.append(self.curr)
            self.curr = 0
            self.nbits = 0

    def to_bytes(self):
        self.finish()
        return bytes(self.buf), self.total_bits

In [ ]:
class BitReader:
    def __init__(self, data_bytes):
        self.data = data_bytes
        self.idx = 0  # bit index

    def read_bit(self):
        byte_i = self.idx // 8
        bit_i = 7 - (self.idx % 8)
        self.idx += 1
        return (self.data[byte_i] >> bit_i) & 1

    def read_bits(self, n):
        v = 0
        for _ in range(n):
            v = (v << 1) | self.read_bit()
        return v

## 6) Rice coding (parameter K)

Rice coding represents an unsigned integer `u` using:
- a **quotient** `q = u >> K` (integer division by `2^K`)
- a **remainder** `r = u & (2^K - 1)` (the lower K bits)

Encoding format:
1. Unary code for `q`: write `q` ones followed by a zero (`111...10`)
2. Write `r` using exactly `K` bits

Decoding reverses this:
- Count 1s until the terminating 0 to recover `q`
- Read the next `K` bits to recover `r`
- Reconstruct `u = (q << K) | r`

I evaluate **K=2** and **K=4** to see which gives better compression for Sound1 and Sound2.

Rice coding is particularly effective when the probability distribution of residuals is geometrically distributed and centred near zero, which is typically the case in predictive audio coding.

The parameter K controls the trade-off between the unary quotient part and the fixed-length remainder.
A smaller value of K produces a shorter remainder but may increase the unary length when the values are large.
A larger value of K increases the number of remainder bits but reduces the unary quotient length for larger values.

In [ ]:
def rice_encode_u(writer, u, k):
    q = u >> k
    r = u & ((1 << k) - 1)

    # FAST unary: write q ones then a zero
    writer.write_ones(q)
    writer.write_bit(0)

    # remainder
    if k > 0:
        writer.write_bits(r, k)

In [ ]:
def rice_decode_u(reader, k):
    q = 0
    while reader.read_bit() == 1:
        q += 1
    r = reader.read_bits(k) if k > 0 else 0
    return (q << k) | r

## 7) Encoder: `.wav → .ex2` (file format + pipeline)

### Encoder pipeline (implemented in `encode_wav_to_ex2`)
For each WAV file:
1. Read WAV metadata and PCM frames.
2. Convert frames → integer samples.
3. Compute residuals using the previous-sample predictor (streamed without storing large arrays).
4. Zigzag-map residuals to unsigned integers (`zz_enc`).
5. Rice-encode each value into a bitstream (`rice_encode_u`).

### `.ex2` file structure (written by this notebook)
The `.ex2` file begins with a small header so decoding is self-contained:
- Magic bytes: `EX2\\0`
- `k` (Rice parameter K)
- `nchannels`, `sampwidth`, `framerate`, `nframes`
- `bit_count` (number of valid bits in the payload)
- Followed by the Rice-coded byte payload

This preserves enough WAV information to reconstruct an identical output WAV.

In [ ]:
def encode_wav_to_ex2(wav_path, ex2_path, k):
    info, frames = read_wav(wav_path)
    samples = bytes_to_samples(frames, info.sampwidth)

    bw = BitWriter()

    # stream residuals without storing arrays
    prev = 0
    for i, s in enumerate(samples):
        if i == 0:
            r = s
        else:
            r = s - prev
        prev = s

        u = zz_enc(r)                 # signed -> unsigned
        rice_encode_u(bw, u, k)       # write bits directly

    payload, bit_count = bw.to_bytes()

    with open(ex2_path, "wb") as f:
        f.write(b"EX2\x00")
        f.write(struct.pack("<B", k))
        f.write(struct.pack("<B", info.nchannels))
        f.write(struct.pack("<B", info.sampwidth))
        f.write(struct.pack("<I", info.framerate))
        f.write(struct.pack("<I", info.nframes))
        f.write(struct.pack("<I", bit_count))
        f.write(payload)

    return os.path.getsize(ex2_path)

## 8) Decoder: `.ex2 → .wav` (lossless reconstruction)

The decoder reads the `.ex2` header to recover:
- K, number of channels, sample width, sample rate, and number of frames.

It then:
1. Rice-decodes `total_samples = nframes × nchannels` unsigned values from the payload.
2. Zigzag-decodes them back to signed residuals (`zz_dec`).
3. Reconstructs the original samples via cumulative summation (`reconstruct_from_residuals`).
4. Converts samples back to PCM bytes and writes a WAV using the original metadata.

Because each stage is exactly invertible, the reconstruction is **lossless**.

In [ ]:
def decode_ex2_to_wav(ex2_path, wav_out_path):
    with open(ex2_path, "rb") as f:
        magic = f.read(4)
        if magic != b"EX2\x00":
            raise ValueError("Not an EX2 file")

        k = struct.unpack("<B", f.read(1))[0]
        nchannels = struct.unpack("<B", f.read(1))[0]
        sampwidth = struct.unpack("<B", f.read(1))[0]
        framerate = struct.unpack("<I", f.read(4))[0]
        nframes = struct.unpack("<I", f.read(4))[0]
        bit_count = struct.unpack("<I", f.read(4))[0]
        payload = f.read()

    br = BitReader(payload)

    total_samples = nframes * nchannels
    uvals = []
    for _ in range(total_samples):
        uvals.append(rice_decode_u(br, k))

    res = [zz_dec(u) for u in uvals]
    samples = reconstruct_from_residuals(res)
    frames = samples_to_bytes(samples, sampwidth)

    with wave.open(wav_out_path, "wb") as w:
        w.setnchannels(nchannels)
        w.setsampwidth(sampwidth)
        w.setframerate(framerate)
        w.writeframes(frames)

## 9) Verification (lossless) + compression evaluation

### Lossless verification
To confirm the codec is truly lossless, I compare the original WAV bytes with the decoded WAV bytes using `files_identical()`.

A result of `True` means:
- The decoded WAV matches the original **byte-for-byte** (not just “sounds similar”).

I run this check for:
- Sound1 with K=4 and K=2
- Sound2 with K=4 and K=2

### Compression table
Finally, I compute:
- Original WAV size (bytes)
- Encoded `.ex2` size for K=4 and K=2
- Compression percentage for each K

This provides an objective comparison of which K performs better on each audio file.

In [ ]:
def files_identical(path_a, path_b):
    with open(path_a, "rb") as fa, open(path_b, "rb") as fb:
        return fa.read() == fb.read()

# Example test for Sound1
encode_wav_to_ex2("Sound1.wav", "Sound1_Enc.ex2", k=4)
decode_ex2_to_wav("Sound1_Enc.ex2", "Sound1_Enc_Dec.wav")
print("Lossless:", files_identical("Sound1.wav", "Sound1_Enc_Dec.wav"))


# --- Additional lossless checks(Sound2 + K=2) ---

# Sound1 with K=2
encode_wav_to_ex2("Sound1.wav", "Sound1_K2_Enc.ex2", k=2)
decode_ex2_to_wav("Sound1_K2_Enc.ex2", "Sound1_K2_Enc_Dec.wav")
print("Sound1 lossless (K=2):", files_identical("Sound1.wav", "Sound1_K2_Enc_Dec.wav"))

# Sound2 with K=4
encode_wav_to_ex2("Sound2.wav", "Sound2_Enc.ex2", k=4)
decode_ex2_to_wav("Sound2_Enc.ex2", "Sound2_Enc_Dec.wav")
print("Sound2 lossless (K=4):", files_identical("Sound2.wav", "Sound2_Enc_Dec.wav"))

# Sound2 with K=2
encode_wav_to_ex2("Sound2.wav", "Sound2_K2_Enc.ex2", k=2)
decode_ex2_to_wav("Sound2_K2_Enc.ex2", "Sound2_K2_Enc_Dec.wav")
print("Sound2 lossless (K=2):", files_identical("Sound2.wav", "Sound2_K2_Enc_Dec.wav"))

## 10) Interpreting differences between Sound1 and Sound2

Compression performance can differ because the residual distribution depends on the audio content:
- If the signal is smoother or more predictable, residuals cluster tightly around 0 → smaller mapped values → better Rice coding.
- If the signal changes rapidly (more transients/noise), residuals have larger magnitudes → larger quotients `q` → longer unary codes → worse compression.

In the report, I will compare Sound1 vs Sound2 and justify why their compression ratios differ under K=2 and K=4.

In [ ]:
def compression_percent(original_size, compressed_size):
    return (1 - (compressed_size / original_size)) * 100

def run_table(sound_path):
    original_size = os.path.getsize(sound_path)

    size_k4 = encode_wav_to_ex2(
        sound_path,
        sound_path.replace(".wav", "_K4_Enc.ex2"),
        k=4
    )

    size_k2 = encode_wav_to_ex2(
        sound_path,
        sound_path.replace(".wav", "_K2_Enc.ex2"),
        k=2
    )

    return {
        "original": original_size,
        "rice_k4": size_k4,
        "rice_k2": size_k2,
        "pct_k4": compression_percent(original_size, size_k4),
        "pct_k2": compression_percent(original_size, size_k2),
    }

import pandas as pd

r1 = run_table("Sound1.wav")
r2 = run_table("Sound2.wav")

df = pd.DataFrame([
    {
        "Sound file": "Sound1.wav",
        "Original size (bytes)": r1["original"],
        "Rice size K=4 (bytes)": r1["rice_k4"],
        "Rice size K=2 (bytes)": r1["rice_k2"],
        "% compression K=4": r1["pct_k4"],
        "% compression K=2": r1["pct_k2"],
        "Encoded file (K=4)": "Sound1_K4_Enc.ex2",
        "Encoded file (K=2)": "Sound1_K2_Enc.ex2",
    },
    {
        "Sound file": "Sound2.wav",
        "Original size (bytes)": r2["original"],
        "Rice size K=4 (bytes)": r2["rice_k4"],
        "Rice size K=2 (bytes)": r2["rice_k2"],
        "% compression K=4": r2["pct_k4"],
        "% compression K=2": r2["pct_k2"],
        "Encoded file (K=4)": "Sound2_K4_Enc.ex2",
        "Encoded file (K=2)": "Sound2_K2_Enc.ex2",
    },
])

# To make it look neat (2 decimal places)
df["% compression K=4"] = df["% compression K=4"].map(lambda x: f"{x:.2f}%")
df["% compression K=2"] = df["% compression K=2"].map(lambda x: f"{x:.2f}%")

df